In [1]:
import os
from langchain_openai import ChatOpenAI
model= ChatOpenAI(
    base_url=os.getenv("DASHSCOPE_API_BASE"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"), 
    # 加载的是环境变量值，如果在配置文件中命名变量名，则会默认使用系统环境变量里的名字,这点需注意！否则秘钥不同会报错的！
    model="qwen-max",
    streaming=True,
)

# Build a basic agent

In [ ]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='778032dc-29d3-45c6-8437-9b5f4a345b20'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 265, 'total_tokens': 286, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen3-max', 'system_fingerprint': None, 'id': 'chatcmpl-3c24d71f-56bb-4e80-ab8e-dc0aaa4606b3', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--b6fb5310-340a-4ae5-b0f7-ed928d2c5510-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'sf'}, 'id': 'call_2c847dfeb5354285b38aa14b', 'type': 'tool_call'}], usage_metadata={'input_tokens': 265, 'output_tokens': 21, 'total_tokens': 286, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
  ToolMessage(content="It's always sunny in sf!", name='get_weath

### Tool Calling

In [5]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

model_with_tools = model.bind_tools([get_weather])  

response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


## Structured output   
文档链接： https://docs.langchain.com/oss/python/langchain/structured-output#response-format

### Response Format 参数
* ToolStrategy[StructuredResponseT]: or models supporting native structured output (e.g. OpenAI, Grok)  一般不使用这种方法，对模型有限制
* ProviderStrategy[StructuredResponseT]: Uses provider-native structured output
* type[StructuredResponseT]: Schema type - automatically selects best strategy based on model capabilities <mark>最好用</mark>
* None: No structured output

In [30]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    # tools=tools,
    response_format=ContactInfo  # ✅ 自动选择最佳策略
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [23]:
result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='085b0df7-c1d4-4ce3-a4ee-cf446978331b'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'qwen-max', 'model_provider': 'openai'}, id='lc_run--b4289901-3b6d-4899-aafe-f092ce45e4cc', tool_calls=[{'name': 'ContactInfo', 'args': {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}, 'id': 'call_7eb056aeaf0040009fd828', 'type': 'tool_call'}]),
  ToolMessage(content="Returning structured response: name='John Doe' email='john@example.com' phone='(555) 123-4567'", name='ContactInfo', id='84fb309a-6f5e-423c-837f-758fbb3fe62b', tool_call_id='call_7eb056aeaf0040009fd828')],
 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}

In [2]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

agent = create_agent(
    model=model,
    response_format=Person  # ✅ 自动选择最佳策略
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "张三是一名 30 岁的软件工程师"}]
})

result["structured_response"]

Person(name='张三', age=30, occupation='软件工程师')

## ToolMessage
对于不支持原生结构化输出的模型，LangChain 通过工具调用（tool calling）来实现相同的效果。此方法适用于所有支持工具调用的模型，而目前大多数现代模型都具备此功能。

In [24]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """Analysis of a product review."""
    rating: int | None = Field(description="The rating of the product", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")
    key_points: list[str] = Field(description="The key points of the review. Lowercase, 1-3 words each.")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ProductReview)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]
# ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

#### 参数：`tool_message_content`:
作用：
**自定义结构化输出在对话历史中显示的内容**   
隐藏内部结构：只保留业务语义，避免在对话历史中暴露字段名、工具名等敏感设计细节。 

In [10]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript."""
    task: str = Field(description="The specific task to be completed")
    assignee: str = Field(description="Person responsible for the task")
    priority: Literal["low", "medium", "high"] = Field(description="Priority level")

agent = create_agent(
    model=model,
    tools=[],  # 虚拟工具
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!"  # 添加到对话历史的消息内容
    )
)

agent.invoke({
    "messages": [{"role": "user", "content": "From our meeting: Sarah needs to update the project timeline as soon as possible"}]
})

{'messages': [HumanMessage(content='From our meeting: Sarah needs to update the project timeline as soon as possible', additional_kwargs={}, response_metadata={}, id='26b38409-b54d-49e4-ab25-a763a2a46233'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'qwen-max', 'model_provider': 'openai'}, id='lc_run--11ce64f9-4e01-4461-9ff3-cd758a7302a3', tool_calls=[{'name': 'MeetingAction', 'args': {'task': 'Update the project timeline', 'assignee': 'Sarah', 'priority': 'high'}, 'id': 'call_1ba18a2904d64f10806ceb', 'type': 'tool_call'}]),
  ToolMessage(content='Action item captured and added to meeting notes!', name='MeetingAction', id='45d9a4dc-a6de-4673-8e99-9db8641b7cc7', tool_call_id='call_1ba18a2904d64f10806ceb')],
 'structured_response': MeetingAction(task='Update the project timeline', assignee='Sarah', priority='high')}

In [16]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductRating(BaseModel):
    rating: int | None = Field(description="Rating from 1-5", ge=1, le=5) # 要求rating字段必须提供，其值要么是 None（表示暂无评分），要么是一个在 1 到 5 之间（包含1和5）的整数。
    # description 不仅是代码注释，更是 给 LLM 的指令，使评分在1-5之间（包含1,5）
    comment: str = Field(description="Review comment")

agent = create_agent(
    model=model,
    tools=[], # 表明这是一个 纯结构化输出任务，不需要真实工具调用,ToolStrategy 会创建内部虚拟工具处理输出格式化
    # response_format=ToolStrategy(ProductRating),  # 默认参数： handle_errors=True
    response_format= ToolStrategy(ProductRating,handle_errors="Please provide a valid rating between 1-5 and include a comment."), # 自定义错误如何处理
    system_prompt="You are a helpful assistant that parses product reviews. Do not make any field or value up."  # 最后一句：不得编造任何字段或值
)

agent.invoke({
    "messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}]
})

{'messages': [HumanMessage(content='Parse this: Amazing product, 10/10!', additional_kwargs={}, response_metadata={}, id='96357499-1836-4ea9-88b4-7f7a2400e52f'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'qwen-max', 'model_provider': 'openai'}, id='lc_run--c45fa3a3-fadd-4b6b-b40a-483be60383b5', tool_calls=[{'name': 'ProductRating', 'args': {'rating': 5, 'comment': 'Amazing product, 10/10!'}, 'id': 'call_04b551f46da54d41be0069', 'type': 'tool_call'}]),
  ToolMessage(content="Returning structured response: rating=5 comment='Amazing product, 10/10!'", name='ProductRating', id='f6e835d1-a64b-46b6-8dd6-d854c3a1e755', tool_call_id='call_04b551f46da54d41be0069')],
 'structured_response': ProductRating(rating=5, comment='Amazing product, 10/10!')}

## 🔧 handle_errors 的五种配置方式
1️⃣ handle_errors = True（默认行为）   **行为：对所有验证错误自动重试**  ✅ 适合简单场景  ❌ 不适合复杂/模糊输入  
2️⃣ handle_errors = "自定义字符串"（固定提示重试） **行为：无论什么错误，都用**同一段固定**提示让模型重试**   
优点：提示更友好、业务相关
缺点：无法区分错误类型，可能误导模型            
适合规则明确的场景（如评分必须 1-5）。    💡 技巧：提示中包含具体约束能显著提升成功率  
3️⃣ handle_errors = ExceptionType（仅特定异常重试）  行为：`handle_errors=ValueError`
如果抛出 `ValueError` → **重试**（使用默认错误消息）
如果抛出其他异常（如 TypeError, RuntimeError）→ **直接抛出，不重试**  
4️⃣ handle_errors = (Exception1, Exception2)（多异常类型重试） 行为：`handle_errors=(ValueError, TypeError)`—— 仅当错误是 ValueError 或 TypeError 时重试   
5️⃣ handle_errors = 自定义函数（完全控制错误处理）    
``` python 
def custom_error_handler(error: Exception) -> str:
    if isinstance(error, StructuredOutputValidationError):
        return "There was an issue with the format. Try again."
    elif isinstance(error, MultipleStructuredOutputsError):
        return "Multiple structured outputs were returned. Pick the most relevant one."
    else:
        return f"Error: {str(error)}"

ToolStrategy(
    schema=Union[ContactInfo, EventDetails],
    handle_errors=custom_error_handler
)  
```
错误类型说明：

|异常类型	|触发场景|	自定义提示示例|
|---|---|---|
|StructuredOutputValidationError|	输出不符合 Pydantic schema	|"格式有问题，请重试"|
|MultipleStructuredOutputsError	|Union 类型下返回多个工具调用	|"请只选择最相关的一个"|
|其他异常|	网络错误、代码 bug 等|	"Error: 具体错误信息"|

6️⃣ handle_errors = False（禁用所有重试）   **行为：任何错误都立即抛出异常，不重试**   调试阶段：快速暴露问题



但是我不太会用上面的`handle_errors`参数啊。。。

In [26]:
from pydantic import BaseModel, Field
from typing import Optional

class ExtractionResult(BaseModel):
    """Extract all possible information"""
    contact_name: Optional[str] = Field(None, description="Person's name if mentioned")
    contact_email: Optional[str] = Field(None, description="Email address if mentioned")
    event_name: Optional[str] = Field(None, description="Event name if mentioned")
    event_date: Optional[str] = Field(None, description="Event date if mentioned")

# 创建 agent
agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(ExtractionResult)
)

# 调用
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

# 输出示例
print(result["structured_response"])


contact_name='John Doe' contact_email='john@email.com' event_name='Tech Conference' event_date='March 15th'


# Build a real-world agent
独特风格约束：who speaks in puns(喜欢说双关语)  
这是一个非常具体且强大的风格控制指令。它要求AI在所有输出中都必须融入与天气相关的文字游戏或幽默双关。例如，当回答晴天时，不能只说“It's sunny”，而可能会说“It's going to be a bright day, not a cloudy one!”。这极大地增强了交互的趣味性和个性，但也对模型的语言生成能力提出了更高要求。

工具能力赋予 (Tool Capability Granting)    
功能扩展: 这部分是现代AI Agent的核心。它向模型宣告了其自身不具备但可以通过外部函数调用获得的能力。这解决了大模型的两个固有局限：**知识过时性（无法获取实时天气）**和**执行能力缺失（无法主动获取用户位置）**。
* get_weather_for_location: 明确指出其用途是“获取特定地点的天气”。
* get_user_location: 明确指出其用途是“获取用户的地理位置”。   

"If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."

核心业务规则: 这段话是整个提示词的“大脑”，它定义了处理“查天气”请求的标准操作流程（SOP）。   
* 第一步：条件判断 - “If a user asks you for the weather...”: 触发条件是用户提出了与天气相关的问题。    
* 第二步：信息完备性检查 - “make sure you know the location.”: 强调了一个关键原则——没有地点信息，就无法提供有效的天气服务。这迫使模型在响应前必须先解决“在哪里”的问题。    
* 第三步：智能推理与动作选择 - “If you can tell from the question that they mean wherever they are...”: 这引入了上下文理解和意图识别。模型需要分析用户的提问方式。
例如：     
用户问：“我这里天气怎么样？” → 隐含了“我的当前位置”，此时应调用 get_user_location。  
用户问：“北京今天下雨吗？” → 地点已明确，无需调用 get_user_location，可直接调用 get_weather_for_location("北京")。  
* 第四步：执行正确动作 - “use the get_user_location tool...”: 在满足条件时，明确指出了应该采取的具体行动。  

In [2]:
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

In [3]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
# tool是一个装饰器，它是定义自定义工具最简单的方式，可以快速将普通函数转换为大语言模型（LLM）能够理解和调用的工具。
# ToolRuntime则是一个泛型类型，用于在工具执行时传递运行时上下文。

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str # str为类型注解

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str: 
    # 当此工具被调用时，LangChain会将包含当前user_id的Context实例注入到runtime对象中。函数体通过runtime.context.user_id提取出用户ID
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

#### @dataclass装饰器
`@dataclass` 是 Python 3.7 引入的一个装饰器，用于简化**数据类的定义**。它可以自动为类生成常用的特殊方法，例如 __init__、__repr__、__eq__ 等，从而减少样板代码的编写，提高代码的可读性。

In [ ]:
from dataclasses import dataclass

# We use a dataclass here, but Pydantic models are also supported.
@dataclass
class ResponseFormat:  # ResponseFormat: 响应格式规范
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

In [ ]:
from langchain.agents.structured_output import ToolStrategy
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,   # 定义对话上下文的数据结构
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

In [9]:
# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])

ResponseFormat(punny_response='Looks like Florida is living up to its rep—sunny side up and not a cloud in sight! Don’t forget the sunscreen, or you’ll be fried like a Floridian alligator at a beach BBQ!', weather_conditions='sunny')


In [10]:
# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])

ResponseFormat(punny_response="You're welcome! Stay cool—or at least as cool as a cucumber in a sauna! 🌞", weather_conditions='sunny')
